In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSVs (they are in current working directory)
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Reusable timezone
tz = pytz.timezone('Asia/Shanghai')

# Helper to convert min/max timestamp (seconds) to tz-aware datetime
def to_tz_datetime_series(s):
    # s is a Series of POSIX seconds (ints)
    return pd.to_datetime(s.astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# AGGREGATION for metric.csv grouping by (cmdb_id, kpi_name)
metric_agg = metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    mean_value=('value', 'mean'),
    std_value=('value', 'std'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
)

# Convert timestamps to Asia/Shanghai tz-aware datetimes
metric_agg['min_timestamp'] = to_tz_datetime_series(metric_agg['min_timestamp'])
metric_agg['max_timestamp'] = to_tz_datetime_series(metric_agg['max_timestamp'])

# Round numeric summaries for compactness
metric_agg[['mean_value', 'std_value', 'p50', 'p95']] = metric_agg[['mean_value', 'std_value', 'p50', 'p95']].round(6)

# Sort by count desc and keep top 20
metric_top20 = metric_agg.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# AGGREGATION for trace.csv grouping by (cmdb_id, trace_name)
trace_agg = trace.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    mean_value=('value', 'mean'),
    std_value=('value', 'std'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
)
trace_agg['min_timestamp'] = to_tz_datetime_series(trace_agg['min_timestamp'])
trace_agg['max_timestamp'] = to_tz_datetime_series(trace_agg['max_timestamp'])
trace_agg[['mean_value', 'std_value', 'p50', 'p95']] = trace_agg[['mean_value', 'std_value', 'p50', 'p95']].round(6)
trace_top20 = trace_agg.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# AGGREGATION for log.csv grouping by (cmdb_id, log_name)
log_agg = log.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    mean_value=('value', 'mean'),
    std_value=('value', 'std'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
)
log_agg['min_timestamp'] = to_tz_datetime_series(log_agg['min_timestamp'])
log_agg['max_timestamp'] = to_tz_datetime_series(log_agg['max_timestamp'])
log_agg[['mean_value', 'std_value', 'p50', 'p95']] = log_agg[['mean_value', 'std_value', 'p50', 'p95']].round(6)
log_top20 = log_agg.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Build deduplicated list of all unique cmdb_id values across all four files
unique_cmdbs = sorted(set(metric['cmdb_id'].unique()) | set(trace['cmdb_id'].unique()) | set(log['cmdb_id'].unique()) | set(error_logs['cmdb_id'].unique()))

# Counts of occurrences per file (rows per component in each file)
metric_counts = metric.groupby('cmdb_id').size().rename('metric_count')
trace_counts = trace.groupby('cmdb_id').size().rename('trace_count')
log_counts = log.groupby('cmdb_id').size().rename('log_count')
error_counts = error_logs.groupby('cmdb_id').size().rename('error_log_count')

# Combine into one DataFrame
comp_counts = pd.concat([metric_counts, trace_counts, log_counts, error_counts], axis=1).fillna(0).astype(int).reset_index().rename(columns={'index':'cmdb_id'})

# Keep a compact display: show full comp_counts (usually small), but it's aggregated counts only
component_counts = comp_counts.sort_values(['metric_count', 'trace_count', 'log_count', 'error_log_count'], ascending=False).reset_index(drop=True)

# Final returned compact objects: top20 aggregates for metric/trace/log and component counts + unique list length
metric_top20, trace_top20, log_top20, component_counts, unique_cmdbs

```
Out[1]:
```
```python
# Compose a plain-English summary string based on previously computed aggregates.
# Reuse variables from earlier execution: metric_top20, trace_top20, log_top20, component_counts, unique_cmdbs

def top_rows_summary(df, name_col, n=3):
    rows = []
    cols = ['cmdb_id', name_col, 'count', 'min_timestamp', 'max_timestamp', 'mean_value', 'std_value', 'p50', 'p95']
    for _, r in df.head(n).iterrows():
        rows.append(
            f"{r['cmdb_id']} | {r[name_col]} | count={int(r['count'])} | "
            f"min={r['min_timestamp']} | max={r['max_timestamp']} | "
            f"mean={r.get('mean_value', '')} std={r.get('std_value','')} p50={r.get('p50','')} p95={r.get('p95','')}"
        )
    return "\n".join(rows)

# Basic totals
num_components = len(unique_cmdbs)
num_comp_rows = component_counts.shape[0]

# Top components by metric_count
top_comps = component_counts.head(5)
top_comps_lines = "; ".join(
    [f"{row.cmdb_id}(metric={row.metric_count}, trace={row.trace_count}, log={row.log_count}, error_log={row.error_log_count})"
     for _, row in top_comps.iterrows()]
)

summary = (
    f"Summary of telemetry aggregates (using full series):\n\n"
    f"1) Unique components: {num_components} (listed). Component-count table has {num_comp_rows} rows.\n"
    f"   Top components by counts: {top_comps_lines}\n\n"
    f"2) Top metric aggregated series (top 3 by count):\n"
    f"{top_rows_summary(metric_top20, 'kpi_name', n=3)}\n\n"
    f"3) Top trace aggregated series (top 3 by count):\n"
    f"{top_rows_summary(trace_top20, 'trace_name', n=3)}\n\n"
    f"4) Top log aggregated series (top 3 by count):\n"
    f"{top_rows_summary(log_top20, 'log_name', n=3)}\n\n"
    f"Notes:\n"
    f"- Metric groups with highest counts had count=31 in the top results (examples shown above).\n"
    f"- Trace top groups mostly have counts around 29 and 28 (examples shown above).\n"
    f"- Log top groups have counts around 27 and 25 for the leading components (examples shown above).\n"
    f"- The full component occurrence table (component -> {{metric_count, trace_count, log_count, error_log_count}}) is available as 'component_counts'.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count             min_timestamp             max_timestamp    mean_value    std_value           p50           p95
0       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  6.945161e+01     1.963648  7.000000e+01  7.250000e+01
1   apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  3.225800e-02     0.179605  0.000000e+00  0.000000e+00
2   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
3    Mysql01                         OSLinux-CPU_CPU_CPUSysTime     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  3.135450e-01     0.075493  2.991030e-01  4.921500e-01
4    Mysql01                        OSLinux-CPU_CPU_CPUUserTime     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  1.626740e-01     0.037864  1.578000e-01  2.409490e-01
5    Mysql01                             OSLinux-CPU_CPU_CPUWio     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  6.030400e-02     0.084860  2.077100e-02  2.596000e-01
6    Mysql01                        OSLinux-CPU_CPU_CPUidleutil     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  9.945866e+01     0.146604  9.950160e+01  9.961165e+01
7    Mysql01                            OSLinux-CPU_CPU_CPULoad     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  3.000000e-02     0.035963  2.000000e-02  1.050000e-01
8   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  2.841904e+08  1970.326688  2.841902e+08  2.841937e+08
9   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  5.358968e+05    28.161936  5.358990e+05  5.359390e+05
10  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
11  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  5.000000e+02     0.000000  5.000000e+02  5.000000e+02
12  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
13  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
14  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
15  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
16  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
17  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
18  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00
19  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00  0.000000e+00     0.000000  0.000000e+00  0.000000e+00,      cmdb_id                       trace_name  count             min_timestamp             max_timestamp  mean_value  std_value        p50         p95
0       IG01         trace.self.duration_mean     29 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.063374   0.090010   0.039000    0.138528
1       IG01          trace.self.duration_p95     29 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.139136   0.259917   0.057650    0.532770
2       IG01             trace.self.row_count     29 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00   12.000000  10.501701  10.000000   31.200000
3       IG02          trace.self.duration_p95     29 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.159955   0.235745   0.070550    0.761780
4       IG02         trace.self.duration_mean     29 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.080396   0.129692   0.037033    0.182220
5       IG02             trace.self.row_count     29 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00   12.000000  10.488088  10.000000   32.400000
6       MG02          trace.self.duration_p95     29 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.120859   0.201753   0.027100    0.598400
7       MG02         trace.self.duration_mean     29 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.036819   0.048181   0.015333    0.153253
8       MG02             trace.self.row_count     29 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    9.517241   9.112866   5.000000   26.600000
9       MG01         trace.self.duration_mean     28 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.030759   0.028208   0.021554    0.075451
10      MG01             trace.self.row_count     28 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00   10.071429   8.571825   6.500000   24.300000
11      MG01          trace.self.duration_p95     28 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.083521   0.118985   0.046450    0.304350
12  dockerB2          trace.self.duration_p95     26 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.020787   0.082210   0.004075    0.010038
13  dockerB2             trace.self.row_count     26 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00   33.615385  45.390816  10.500000  137.750000
14  dockerB2         trace.self.duration_mean     26 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.004358   0.014978   0.001000    0.009055
15  Tomcat04             trace.self.row_count     25 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00   27.120000  25.292160  18.000000   70.200000
16      MG02   trace.to_dockerB2.duration_p95     25 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.117722   0.285836   0.006000    0.760160
17      MG02  trace.to_dockerB2.duration_mean     25 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.075917   0.181010   0.006000    0.388650
18      MG02      trace.to_dockerB2.row_count     25 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    2.320000   1.405940   2.000000    4.000000
19  Tomcat04         trace.self.duration_mean     25 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.013032   0.025637   0.004472    0.058659,      cmdb_id          log_name  count             min_timestamp             max_timestamp  mean_value  std_value    p50      p95
0   apache01     log.row_count     27 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    6.148148   5.267489  5.000  16.7000
1   apache01   log.error_count     27 2021-03-07 03:00:00+08:00 2021-03-07 03:29:00+08:00    0.000000   0.000000  0.000   0.0000
2   apache02   log.error_count     27 2021-03-07 03:01:00+08:00 2021-03-07 03:30:00+08:00    0.000000   0.000000  0.000   0.0000
3   apache02     log.row_count     27 2021-03-07 03:01:00+08:00 2021-03-07 03:30:00+08:00    6.777778   5.279374  5.000  15.7000
4   Tomcat03     log.row_count     25 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00    4.040000   3.155419  4.000   7.8000
5   Tomcat04     log.row_count     25 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    4.080000   3.534591  3.000  11.6000
6   Tomcat01   log.error_count     25 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.000000   0.000000  0.000   0.0000
7   Tomcat01     log.row_count     25 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    2.960000   3.181719  2.000   9.8000
8   Tomcat03   log.error_count     25 2021-03-07 03:00:00+08:00 2021-03-07 03:30:00+08:00    0.000000   0.000000  0.000   0.0000
9   Tomcat04   log.error_count     25 2021-03-07 03:01:00+08:00 2021-03-07 03:29:00+08:00    0.000000   0.000000  0.000   0.0000
10  Tomcat02   log.error_count     22 2021-03-07 03:01:00+08:00 2021-03-07 03:28:00+08:00    0.000000   0.000000  0.000   0.0000
11  Tomcat02     log.row_count     22 2021-03-07 03:01:00+08:00 2021-03-07 03:28:00+08:00    3.636364   3.016549  2.000   8.9500
12  Tomcat03  log.gc_pause_max      2 2021-03-07 03:08:00+08:00 2021-03-07 03:24:00+08:00    0.060000   0.014142  0.060   0.0690
13  Tomcat02  log.gc_pause_max      2 2021-03-07 03:10:00+08:00 2021-03-07 03:20:00+08:00    0.080000   0.000000  0.080   0.0800
14  Tomcat01  log.gc_pause_max      2 2021-03-07 03:03:00+08:00 2021-03-07 03:25:00+08:00    0.080000   0.000000  0.080   0.0800
15  Tomcat04  log.gc_pause_max      2 2021-03-07 03:03:00+08:00 2021-03-07 03:22:00+08:00    0.065000   0.007071  0.065   0.0695
16   Mysql01     log.row_count      1 2021-03-07 03:24:00+08:00 2021-03-07 03:24:00+08:00    1.000000        NaN  1.000   1.0000
17   Mysql01   log.error_count      1 2021-03-07 03:24:00+08:00 2021-03-07 03:24:00+08:00    0.000000        NaN  0.000   0.0000,          cmdb_id  metric_count  trace_count  log_count  error_log_count
0        Mysql01          4116            0          2                0
1        Mysql02          4091            0          0                0
2        Redis01          2290            0          0                0
3        Redis02          2246            0          0                0
4       Tomcat02          1620          279         46                0
..           ...           ...          ...        ...              ...
23  ServiceTest2            12            0          0                0
24  ServiceTest7            12            0          0                0
25  ServiceTest9            12            0          0                0
26      dockerB2            10          267          0                0
27      dockerA1             6           72          0                0

[28 rows x 5 columns], ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'metric' DataFrame from prior execution
# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (no filtering)
global_p95 = metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})

# 2) Filter metric.csv to the incident window (inclusive) in UTC seconds.
# Window given in Asia/Shanghai (UTC+8)
start_local = pd.Timestamp('2021-03-07 03:00:00', tz=tz)
end_local   = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
# Convert to UTC epoch seconds to match metric.timestamp (which is epoch seconds)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

window_df = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 3) Join windowed rows with global_p95 and flag anomalies where value >= global_p95
# Merge on cmdb_id + kpi_name
window_with_p95 = window_df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Drop rows where global_p95 is NaN (shouldn't normally happen) then detect anomalies
window_with_p95 = window_with_p95.dropna(subset=['global_p95'])
anomalies = window_with_p95[window_with_p95['value'] >= window_with_p95['global_p95']].copy()

# 4A) Component-KPI anomaly summary
if not anomalies.empty:
    agg_a = anomalies.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        anomaly_count=('value', 'size'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
    # severity_ratio, handle division by zero
    agg_a['severity_ratio'] = np.where(agg_a['global_p95'] == 0,
                                       np.nan,
                                       agg_a['max_value_in_window'] / agg_a['global_p95'])
    # Convert earliest_anomaly_timestamp to tz-aware datetime for readability
    agg_a['earliest_anomaly_timestamp'] = pd.to_datetime(agg_a['earliest_anomaly_timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    # Round numeric columns for compactness
    agg_a[['max_value_in_window', 'global_p95', 'severity_ratio']] = agg_a[['max_value_in_window', 'global_p95', 'severity_ratio']].round(6)
    # Sort by anomaly_count desc and keep top 20
    anomaly_summary = agg_a.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    anomaly_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity_ratio'])

# 4B) Per-component anomaly summary
if not anomalies.empty:
    agg_b = anomalies.groupby('cmdb_id', as_index=False).agg(
        total_anomaly_count=('value', 'size'),
        earliest_anomaly_timestamp=('timestamp', 'min')
    )
    agg_b['earliest_anomaly_timestamp'] = pd.to_datetime(agg_b['earliest_anomaly_timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    component_anomaly_summary = agg_b.sort_values('total_anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    component_anomaly_summary = pd.DataFrame(columns=['cmdb_id','total_anomaly_count','earliest_anomaly_timestamp'])

# Display the two compact tables (limited to top 20 as requested)
anomaly_summary.head(20), component_anomaly_summary.head(20)

```
Out[2]:
```
```python
# Compose a concise plain-English summary string from prior results.
# Reuse existing variables: anomaly_summary, component_anomaly_summary
tz_name = 'Asia/Shanghai'

top_comp_rows = component_anomaly_summary.head(10)
top_comp_list = "; ".join(
    [f"{r.cmdb_id}(count={int(r.total_anomaly_count)}, earliest={r.earliest_anomaly_timestamp})"
     for _, r in top_comp_rows.iterrows()]
)

top_kpi_rows = anomaly_summary.head(5)
top_kpi_list = []
for _, r in top_kpi_rows.iterrows():
    # format numeric fields succinctly
    max_val = r['max_value_in_window']
    p95 = r['global_p95']
    sev = r['severity_ratio']
    sev_str = "NaN" if pd.isna(sev) else f"{sev:.3f}"
    top_kpi_list.append(
        f"{r.cmdb_id} | {r.kpi_name} | anomalies={int(r.anomaly_count)} | earliest={r.earliest_anomaly_timestamp} | max={max_val:.6g} | p95={p95:.6g} | severity={sev_str}"
    )
top_kpi_text = "\n".join(top_kpi_list)

final_summary = (
    "Anomaly detection summary for metric.csv (using global P95 thresholds computed from full series):\n\n"
    f"- Components with anomalies: {len(component_anomaly_summary)} (shown top 10): {top_comp_list}\n\n"
    "- Example component-KPI anomaly entries (top 5 by anomaly count):\n"
    f"{top_kpi_text}\n\n"
    "Notes:\n"
    "- Many component-KPI groups show anomaly_count=31 in the incident window (examples above).\n"
    "- For several KPI groups global_p95 == max_value_in_window, yielding severity_ratio ~= 1.0 (exact matches to P95).\n"
    "- Some KPI groups have global_p95 == 0 which produces undefined (NaN) severity_ratio; these appear in the top anomalies list.\n"
    "- The most affected components by total anomalies (top 14) are: "
    + ", ".join(list(component_anomaly_summary.head(14)['cmdb_id'].astype(str)))
    + "."
)

final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95  severity_ratio
0   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
1   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-07 03:00:00+08:00         1.000000e+02  1.000000e+02             1.0
2   apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-07 03:00:00+08:00         4.000000e-01  4.000000e-01             1.0
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-07 03:00:00+08:00         5.796209e+07  5.796209e+07             1.0
4   apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-07 03:00:00+08:00         1.000000e+00  0.000000e+00             NaN
5   Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
6   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31  2021-03-07 03:00:00+08:00         3.890000e+02  3.890000e+02             1.0
7   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-07 03:00:00+08:00         3.991480e+05  3.991480e+05             1.0
8   Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
9   Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
10  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
11      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-07 03:00:00+08:00         1.000000e+00  1.000000e+00             1.0
12  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
13      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-07 03:00:00+08:00         3.000000e-01  3.000000e-01             1.0
14      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-07 03:00:00+08:00         3.000000e+00  3.000000e+00             1.0
15      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-07 03:00:00+08:00         2.040136e+09  2.040136e+09             1.0
16      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-07 03:00:00+08:00         5.803958e+07  5.803958e+07             1.0
17  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
18   Redis02    redis-Redis_6379_Redis  (used_cpu_sys_children)             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN
19   Redis02   redis-Redis_6379_Redis  (used_cpu_user_children)             31  2021-03-07 03:00:00+08:00         0.000000e+00  0.000000e+00             NaN,          cmdb_id  total_anomaly_count earliest_anomaly_timestamp
0        Mysql01                 2649  2021-03-07 03:00:00+08:00
1        Mysql02                 2576  2021-03-07 03:00:00+08:00
2        Redis02                 1421  2021-03-07 03:00:00+08:00
3        Redis01                 1373  2021-03-07 03:00:00+08:00
4       Tomcat02                 1125  2021-03-07 03:00:00+08:00
5       Tomcat04                 1096  2021-03-07 03:00:00+08:00
6       Tomcat03                 1094  2021-03-07 03:00:00+08:00
7       Tomcat01                  937  2021-03-07 03:00:00+08:00
8           IG02                  739  2021-03-07 03:00:00+08:00
9       apache02                  724  2021-03-07 03:00:00+08:00
10          IG01                  703  2021-03-07 03:00:00+08:00
11          MG02                  616  2021-03-07 03:00:00+08:00
12          MG01                  591  2021-03-07 03:00:00+08:00
13      apache01                  529  2021-03-07 03:00:00+08:00
14  ServiceTest5                   28  2021-03-07 03:04:00+08:00
15  ServiceTest6                   24  2021-03-07 03:03:00+08:00
16  ServiceTest8                   23  2021-03-07 03:03:00+08:00
17  ServiceTest3                   22  2021-03-07 03:07:00+08:00
18      dockerA2                   21  2021-03-07 03:02:00+08:00
19  ServiceTest1                   16  2021-03-07 03:00:00+08:00)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables: metric, global_p95, component_anomaly_summary if available.
# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Recompute incident window timestamps (UTC epoch seconds) to be safe
start_local = pd.Timestamp('2021-03-07 03:00:00', tz=tz)
end_local   = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

# 1) Top 10 components by total_anomaly_count from existing component_anomaly_summary
top10_components = component_anomaly_summary.sort_values('total_anomaly_count', ascending=False).head(10)['cmdb_id'].astype(str).tolist()

# 2) Filter metric to incident window and to top10 components
window_metric = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts) & (metric['cmdb_id'].isin(top10_components))].copy()

# 3) Merge global_p95 thresholds (precomputed globally) and mark anomalies (value >= global_p95)
# Ensure global_p95 exists
merged = window_metric.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
# Drop rows without a threshold (should be rare)
merged = merged.dropna(subset=['global_p95']).copy()
merged['is_anomaly'] = merged['value'] >= merged['global_p95']

# Keep only anomalous rows for run detection
anoms = merged[merged['is_anomaly']].copy()
if anoms.empty:
    fault_runs = pd.DataFrame(columns=['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                                       'duration_minutes','points_in_fault','max_value_in_fault','global_p95','severity_ratio'])
else:
    # 4) Collapse consecutive anomalous timestamps into fault runs per (cmdb_id,kpi_name)
    # Sort by keys
    anoms = anoms.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
    # Compute time diff from previous row within each group
    anoms['prev_ts'] = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
    anoms['ts_diff'] = anoms['timestamp'] - anoms['prev_ts']
    # A new run starts when ts_diff != 60 or prev is NA
    anoms['new_run'] = (anoms['ts_diff'] != 60) | (anoms['ts_diff'].isna())
    # Run id as cumulative sum of new_run within group
    anoms['run_id'] = anoms.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()

    # Aggregate each run
    agg = anoms.groupby(['cmdb_id','kpi_name','run_id'], as_index=False).agg(
        fault_start_ts=('timestamp','min'),
        fault_end_ts=('timestamp','max'),
        points_in_fault=('timestamp','size'),
        max_value_in_fault=('value','max'),
        global_p95=('global_p95','first')
    )
    # duration in minutes (inclusive)
    agg['duration_minutes'] = ((agg['fault_end_ts'] - agg['fault_start_ts'])//60) + 1

    # severity_ratio handling division by zero
    agg['severity_ratio'] = np.where(agg['global_p95'] == 0, np.nan, agg['max_value_in_fault'] / agg['global_p95'])

    # Convert timestamps to tz-aware datetimes for readability
    agg['fault_start_timestamp'] = pd.to_datetime(agg['fault_start_ts'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    agg['fault_end_timestamp']   = pd.to_datetime(agg['fault_end_ts'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

    # Select and order columns as requested
    fault_runs = agg[['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                      'duration_minutes','points_in_fault','max_value_in_fault','global_p95','severity_ratio']].copy()

    # Round numeric columns for compactness
    fault_runs[['max_value_in_fault','global_p95','severity_ratio']] = fault_runs[['max_value_in_fault','global_p95','severity_ratio']].round(6)

# 5) Return top 20 fault runs across these top-10 components sorted by severity_ratio desc then earliest fault_start asc
# severity_ratio NaN should be treated as lowest; sort accordingly
fault_runs_sorted = fault_runs.assign(
    _sev_null = fault_runs['severity_ratio'].isna()
).sort_values(by=['_sev_null','severity_ratio','fault_start_timestamp'], ascending=[True, False, True]).drop(columns=['_sev_null'])

fault_runs_top20 = fault_runs_sorted.head(20).reset_index(drop=True)

# Display the compact table (selected columns only)
fault_runs_top20

```
Out[3]:
```
# Compose a concise plain-English summary based on the fault_runs_top20 table computed in the kernel.
# Reuse variable 'fault_runs_top20' from prior execution.
import pandas as pd

if 'fault_runs_top20' not in globals() or fault_runs_top20.empty:
    summary = "No fault runs found for the selected top-10 components in the incident window."
else:
    df = fault_runs_top20.copy()
    # Basic counts and top components
    total_runs = df.shape[0]
    top_components = df['cmdb_id'].value_counts().head(5)
    top_comps_text = ", ".join([f"{c}({n})" for c, n in top_components.items()])

    # Compose top 5 fault-run lines
    lines = []
    for i, row in df.head(5).iterrows():
        sev = row['severity_ratio']
        sev_str = "NaN" if pd.isna(sev) else f"{sev:.3f}"
        lines.append(
            f"{i+1}) {row['cmdb_id']} | {row['kpi_name']} | "
            f"{row['fault_start_timestamp']} -> {row['fault_end_timestamp']} | "
            f"{int(row['duration_minutes'])} min | points={int(row['points_in_fault'])} | "
            f"max={row['max_value_in_fault']:.6g} | p95={row['global_p95']:.6g} | severity={sev_str}"
        )

    summary = (
        f"Top fault-run summary (top {total_runs} runs returned):\n\n"
        f"- Most frequent components in these runs: {top_comps_text}\n\n"
        f"- Example top fault runs:\n" + "\n".join(lines) + "\n\n"
        f"Key observations:\n"
        f"- The most severe runs (highest severity_ratio) affect Mysql01 and Mysql02 (largest ratios >3x and up to ~9.7x).\n"
        f"- Many fault runs are short (1–2 minutes) and concentrated on MySQL-related KPIs (I/O, rows read, buffer pool, data writes).\n"
        f"- A few non-MySQL entries appear (e.g., Tomcat02 CPU WIO) but MySQL dominates the top severity-ranked runs.\n"
        f"- Runs where global_p95 == 0 produce undefined severity (shown as NaN) and should be reviewed separately.\n"
        f"\nUse the variable 'fault_runs_top20' for the compact table of these runs."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name     fault_start_timestamp       fault_end_timestamp  duration_minutes  points_in_fault  max_value_in_fault  global_p95  severity_ratio
0    Mysql01                         Mysql-MySQL_3306_Sort Scan 2021-03-07 03:11:00+08:00 2021-03-07 03:12:00+08:00                 2                2            0.163900     0.01690        9.698225
1    Mysql01                         Mysql-MySQL_3306_Rows Read 2021-03-07 03:11:00+08:00 2021-03-07 03:12:00+08:00                 2                2           87.754100     9.27120        9.465236
2    Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages flushed 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.783300     0.20000        8.916500
3    Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.783300     0.20000        8.916500
4    Mysql01              Mysql-MySQL_3306_Innodb pages written 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.783300     0.20000        8.916500
5    Mysql01               Mysql-MySQL_3306_Innodb data written 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1        58470.400000  6826.66670        8.565000
6    Mysql02                 Mysql-MySQL_3306_Handler Read Prev 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1           19.433300     3.76670        5.159238
7    Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps 2021-03-07 03:28:00+08:00 2021-03-07 03:28:00+08:00                 1                1            5.000000     1.25000        4.000000
8   Tomcat02                             OSLinux-CPU_CPU_CPUWio 2021-03-07 03:17:00+08:00 2021-03-07 03:17:00+08:00                 1                1            0.195300     0.04941        3.952641
9    Mysql01         Mysql-MySQL_3306_Innodb log write requests 2021-03-07 03:00:00+08:00 2021-03-07 03:00:00+08:00                 1                1            2.733300     0.70000        3.904714
10   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-07 03:02:00+08:00 2021-03-07 03:02:00+08:00                 1                1            5.000000     1.50000        3.333333
11   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-07 03:15:00+08:00 2021-03-07 03:15:00+08:00                 1                1            0.800000     0.24000        3.333333
12   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.016667     0.00500        3.333333
13   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.133333     0.04000        3.333333
14   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.016700     0.00501        3.333333
15   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.133300     0.03999        3.333333
16   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-07 03:27:00+08:00 2021-03-07 03:27:00+08:00                 1                1            0.533300     0.15999        3.333333
17   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-07 03:27:00+08:00 2021-03-07 03:27:00+08:00                 1                1            4.266700     1.28001        3.333333
18   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPer... 2021-03-07 03:29:00+08:00 2021-03-07 03:29:00+08:00                 1                1            1.200000     0.36000        3.333333
19   Mysql01                Mysql-MySQL_3306_Innodb data writes 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.983300     0.60000        3.305500```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'trace' DataFrame from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, trace_name) using full series
trace_global_p95 = trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})

# 2) Incident window (inclusive) in UTC+8 -> convert to epoch seconds (UTC)
start_local = pd.Timestamp('2021-03-07 03:00:00', tz=tz)
end_local   = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and detect anomalies (value >= global_p95)
trace_window = trace_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')
trace_window = trace_window.dropna(subset=['global_p95']).copy()
trace_window['is_anomaly'] = trace_window['value'] >= trace_window['global_p95']

anomalies = trace_window[trace_window['is_anomaly']].copy()

# 4) Aggregate per component-trace_name
if anomalies.empty:
    trace_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','global_p95','severity_ratio'
    ])
else:
    agg = anomalies.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        anomaly_count=('value','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    )
    # severity ratio, handle division by zero
    agg['severity_ratio'] = np.where(agg['global_p95'] == 0, np.nan, agg['max_value_in_window'] / agg['global_p95'])
    # convert earliest timestamp to tz-aware datetime (Asia/Shanghai)
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    # round numeric columns
    agg[['max_value_in_window','global_p95','severity_ratio']] = agg[['max_value_in_window','global_p95','severity_ratio']].round(6)
    # 5) sort by earliest_anomaly_timestamp asc then anomaly_count desc, limit top 20
    trace_anomaly_summary = agg.sort_values(['earliest_anomaly_timestamp','anomaly_count'], ascending=[True, False]).head(20).reset_index(drop=True)

# Display the compact aggregated table (top 20)
trace_anomaly_summary

```
Out[4]:
```
```python
# Summarize trace anomaly scan results in plain English using existing variable 'trace_anomaly_summary'.
# Display the summary string as the final output.
if 'trace_anomaly_summary' not in globals() or trace_anomaly_summary.empty:
    summary = "No trace anomalies detected in the incident window."
else:
    df = trace_anomaly_summary.copy()
    total_rows = df.shape[0]
    # Top by anomaly_count
    top_by_count = df.sort_values('anomaly_count', ascending=False).head(3)
    top_list = []
    for _, r in top_by_count.iterrows():
        top_list.append(f"{r['cmdb_id']} | {r['trace_name']} (count={int(r['anomaly_count'])}, earliest={r['earliest_anomaly_timestamp']}, severity={r['severity_ratio']:.3f})")
    # Highest severity
    df_sev = df.dropna(subset=['severity_ratio']).sort_values('severity_ratio', ascending=False)
    highest_sev = df_sev.head(1)
    highest_sev_text = ""
    if not highest_sev.empty:
        r = highest_sev.iloc[0]
        highest_sev_text = f"{r['cmdb_id']} | {r['trace_name']} (severity={r['severity_ratio']:.3f}, max={r['max_value_in_window']}, p95={r['global_p95']})"
    # Compose summary
    summary = (
        f"Trace anomaly scan (incident window 2021-03-07 03:00–03:30 +08:00):\n\n"
        f"- Aggregate results returned {total_rows} component-trace anomaly rows (top 20 shown).\n\n"
        f"- Most prominent anomalies by count (examples):\n  " + "\n  ".join(top_list) + "\n\n"
        f"- Highest severity anomaly (largest max/p95 ratio):\n  {highest_sev_text}\n\n"
        f"- Observations:\n"
        f"  - The strongest signal is between MG02 and dockerB2 (trace.to_dockerB2.row_count and trace.from_MG02.row_count), each with 8 anomaly points starting ~2021-03-07 03:02+08:00.\n"
        f"  - IG01 and IG02 show elevated trace.self.row_count around 03:04–03:05 with severity ratios ~1.30–1.35 (notable increases in self row counts).\n"
        f"  - Most anomalies are small-to-moderate exceedances of the global P95 (severity ratios generally ~1.02–1.37), indicating values modestly above historical 95th percentiles.\n"
        f"  - Many anomalies are 'row_count' (traffic/throughput) related; a few are duration/p95 metrics with small increases.\n\n"
        f"Recommendation: investigate MG02 <-> dockerB2 interactions and the IG nodes (IG01/IG02) for increased request traffic or routing changes around 03:02–03:05; duration anomalies are minor but worth checking if they correlate with traffic spikes."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity_ratio
0       MG02        trace.to_dockerB2.row_count              8  2021-03-07 03:02:00+08:00               5.0000     4.00000        1.250000
1   dockerB2          trace.from_MG02.row_count              8  2021-03-07 03:02:00+08:00               5.0000     4.00000        1.250000
2       IG01               trace.self.row_count              2  2021-03-07 03:04:00+08:00              42.0000    31.20000        1.346154
3       IG02        trace.to_Tomcat02.row_count              2  2021-03-07 03:04:00+08:00               4.0000     4.00000        1.000000
4       MG01               trace.self.row_count              2  2021-03-07 03:04:00+08:00              29.0000    24.30000        1.193416
5   Tomcat02          trace.from_IG02.row_count              2  2021-03-07 03:04:00+08:00               4.0000     4.00000        1.000000
6       MG01      trace.from_Tomcat02.row_count              1  2021-03-07 03:04:00+08:00               7.0000     5.10000        1.372549
7       MG01     trace.to_dockerA2.duration_p95              1  2021-03-07 03:04:00+08:00               0.0770     0.07525        1.023256
8       MG01        trace.to_dockerA2.row_count              1  2021-03-07 03:04:00+08:00               8.0000     6.60000        1.212121
9       MG02  trace.from_dockerB1.duration_mean              1  2021-03-07 03:04:00+08:00               0.0100     0.00850        1.176471
10      MG02   trace.from_dockerB1.duration_p95              1  2021-03-07 03:04:00+08:00               0.0109     0.00980        1.112245
11  Tomcat02            trace.to_MG01.row_count              1  2021-03-07 03:04:00+08:00               7.0000     5.10000        1.372549
12  dockerA2       trace.from_MG01.duration_p95              1  2021-03-07 03:04:00+08:00               0.0770     0.07525        1.023256
13  dockerA2          trace.from_MG01.row_count              1  2021-03-07 03:04:00+08:00               8.0000     6.60000        1.212121
14  dockerB1        trace.to_MG02.duration_mean              1  2021-03-07 03:04:00+08:00               0.0100     0.00850        1.176471
15  dockerB1         trace.to_MG02.duration_p95              1  2021-03-07 03:04:00+08:00               0.0109     0.00980        1.112245
16      IG01        trace.to_Tomcat04.row_count              2  2021-03-07 03:05:00+08:00               6.0000     6.00000        1.000000
17      IG02               trace.self.row_count              2  2021-03-07 03:05:00+08:00              42.0000    32.40000        1.296296
18      IG02        trace.to_Tomcat01.row_count              2  2021-03-07 03:05:00+08:00               6.0000     6.00000        1.000000
19      MG02      trace.from_Tomcat01.row_count              2  2021-03-07 03:05:00+08:00               6.0000     6.00000        1.000000```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing DataFrames 'log' and 'error_logs' from kernel.
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, log_name) using full series (thresholds BEFORE filtering)
log_global_p95 = log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})

# 2) Incident window boundaries (UTC epoch seconds)
start_local = pd.Timestamp('2021-03-07 03:00:00', tz=tz)
end_local   = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

# Filter log.csv to the incident window
log_window = log[(log['timestamp'] >= start_ts) & (log['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and detect anomalies (value >= global_p95)
log_window = log_window.merge(log_global_p95, on=['cmdb_id', 'log_name'], how='left')
# Drop rows without a precomputed threshold
log_window = log_window.dropna(subset=['global_p95']).copy()
log_window['is_anomaly'] = log_window['value'] >= log_window['global_p95']

log_anoms = log_window[log_window['is_anomaly']].copy()

# Aggregate anomaly summaries per (cmdb_id, log_name)
if log_anoms.empty:
    log_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','global_p95','severity_ratio'
    ])
else:
    agg = log_anoms.groupby(['cmdb_id','log_name'], as_index=False).agg(
        anomaly_count=('value','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    )
    agg['severity_ratio'] = np.where(agg['global_p95'] == 0, np.nan, agg['max_value_in_window'] / agg['global_p95'])
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    agg[['max_value_in_window','global_p95','severity_ratio']] = agg[['max_value_in_window','global_p95','severity_ratio']].round(6)
    # Sort by anomaly_count desc then earliest_anomaly_timestamp asc, limit top 20
    log_anomaly_summary = agg.sort_values(['anomaly_count','earliest_anomaly_timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)

# 4) error_logs: filter to same incident window and produce counts + samples
err_window = error_logs[(error_logs['timestamp'] >= start_ts) & (error_logs['timestamp'] <= end_ts)].copy()
if not err_window.empty:
    # Counts per component (only those with entries), sorted desc
    error_counts_table = err_window.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'error_count'}).sort_values('error_count', ascending=False).reset_index(drop=True)
    # Convert timestamps for sampling display
    err_window['timestamp_dt'] = pd.to_datetime(err_window['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    # Prioritize Mysql01 and Mysql02: include all their rows first (if present), then fill remaining slots with earliest rows from other components.
    mysql_rows = err_window[err_window['cmdb_id'].isin(['Mysql01','Mysql02'])].copy()
    other_rows = err_window[~err_window['cmdb_id'].isin(['Mysql01','Mysql02'])].copy().sort_values('timestamp')
    # Build sample list (limit total to 20)
    samples_list = []
    # Include mysql rows first (preserve their order by timestamp)
    if not mysql_rows.empty:
        mysql_rows = mysql_rows.sort_values('timestamp')
        samples_list.append(mysql_rows)
    # Fill the rest with earliest other rows until reaching 20
    remaining_slots = 20 - (mysql_rows.shape[0] if not mysql_rows.empty else 0)
    if remaining_slots > 0 and not other_rows.empty:
        samples_list.append(other_rows.head(remaining_slots))
    if samples_list:
        error_samples = pd.concat(samples_list, ignore_index=True)
        # Limit to required columns and keep ordering by timestamp
        error_samples = error_samples.sort_values('timestamp').head(20).loc[:, ['timestamp_dt','cmdb_id','message']].rename(columns={'timestamp_dt':'timestamp'})
    else:
        error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
else:
    error_counts_table = pd.DataFrame(columns=['cmdb_id','error_count'])
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Final compact outputs (limited aggregates only)
log_anomaly_summary, error_counts_table, error_samples

```
Out[5]:
```
```python
# Compose a concise plain-English summary based on existing results:
# Reuse variables: log_anomaly_summary, error_counts_table, error_samples
if 'log_anomaly_summary' not in globals() or log_anomaly_summary.empty:
    summary = "No log anomalies detected in the incident window."
else:
    df = log_anomaly_summary.copy()
    # Examples: top row_count spikes by severity (exclude NaN severities)
    row_count_examples = df[df['log_name'].str.contains('row_count', na=False)].sort_values('severity_ratio', ascending=False).head(5)
    top_rows = []
    for _, r in row_count_examples.iterrows():
        top_rows.append(f"{r['cmdb_id']} | {r['log_name']} | count={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp']} | max={r['max_value_in_window']} | p95={r['global_p95']} | severity={r['severity_ratio']:.3f}")
    # Also show topGc or other notable entries
    other_examples = df[~df['log_name'].str.contains('row_count', na=False)].sort_values('severity_ratio', ascending=False).head(3)
    other_rows = []
    for _, r in other_examples.iterrows():
        sev = "NaN" if pd.isna(r['severity_ratio']) else f"{r['severity_ratio']:.3f}"
        other_rows.append(f"{r['cmdb_id']} | {r['log_name']} | count={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp']} | max={r['max_value_in_window']} | p95={r['global_p95']} | severity={sev}")

    # Error logs summary
    if 'error_counts_table' not in globals() or error_counts_table.empty:
        err_counts_text = "No error_log entries found in the incident window."
        err_samples_text = ""
    else:
        # compact counts table text
        counts_preview = ", ".join([f"{row.cmdb_id}({int(row.error_count)})" for _, row in error_counts_table.iterrows()])
        err_counts_text = f"Error log counts in window (component -> count): {counts_preview}"
        if 'error_samples' in globals() and not error_samples.empty:
            # show up to 6 sample descriptors to keep summary concise
            samp = error_samples.head(6)
            samp_lines = [f"{r.timestamp} | {r.cmdb_id} | {str(r.message)[:80]}" for _, r in samp.iterrows()]
            err_samples_text = " Example error log rows (timestamp | cmdb_id | message snippet):\n  " + "\n  ".join(samp_lines)
        else:
            err_samples_text = " No example error log rows to show."

    summary = (
        "Bounded log analysis (incident window 2021-03-07 03:00–03:30 +08:00):\n\n"
        "1) Log anomaly summary (top examples):\n"
        f"- row_count spikes (examples):\n  " + ("\n  ".join(top_rows) if top_rows else "None") + "\n\n"
        f"- other notable log anomalies (examples):\n  " + ("\n  ".join(other_rows) if other_rows else "None") + "\n\n"
        "Observations:\n"
        "- Several components (apache01/apache02, Tomcat01–04) show many log.error_count anomalies, but these have max=0 and global_p95=0 -> severity undefined (NaN); these likely reflect zero-valued event counters and need special handling.\n"
        "- Meaningful spikes are in log.row_count for Tomcat and Apache nodes (e.g., Tomcat03, Tomcat01, apache02, Tomcat02, apache01) with severity ratios ~1.34–1.92, indicating traffic/row-count increases above historical P95.\n"
        "- A few gc_pause_max anomalies appear with mild exceedances (~1.00–1.02).\n\n"
        "2) Error logs in the window:\n"
        f"- {err_counts_text}\n"
        f"{err_samples_text}\n\n"
        "Recommendation: investigate the row_count spikes on Tomcat/Apache (they show clear exceedances of P95). Treat log.error_count groups with zero thresholds carefully (they produce NaN severity) — verify instrumentation or thresholding for those counters.\n"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  severity_ratio
0   apache01   log.error_count             27  2021-03-07 03:00:00+08:00                 0.00      0.0000             NaN
1   apache02   log.error_count             27  2021-03-07 03:01:00+08:00                 0.00      0.0000             NaN
2   Tomcat03   log.error_count             25  2021-03-07 03:00:00+08:00                 0.00      0.0000             NaN
3   Tomcat01   log.error_count             25  2021-03-07 03:01:00+08:00                 0.00      0.0000             NaN
4   Tomcat04   log.error_count             25  2021-03-07 03:01:00+08:00                 0.00      0.0000             NaN
5   Tomcat02   log.error_count             22  2021-03-07 03:01:00+08:00                 0.00      0.0000             NaN
6   Tomcat01  log.gc_pause_max              2  2021-03-07 03:03:00+08:00                 0.08      0.0800        1.000000
7   Tomcat02     log.row_count              2  2021-03-07 03:04:00+08:00                12.00      8.9500        1.340782
8   apache02     log.row_count              2  2021-03-07 03:04:00+08:00                22.00     15.7000        1.401274
9   Tomcat01     log.row_count              2  2021-03-07 03:05:00+08:00                14.00      9.8000        1.428571
10  Tomcat04     log.row_count              2  2021-03-07 03:05:00+08:00                13.00     11.6000        1.120690
11  apache01     log.row_count              2  2021-03-07 03:05:00+08:00                20.00     16.7000        1.197605
12  Tomcat02  log.gc_pause_max              2  2021-03-07 03:10:00+08:00                 0.08      0.0800        1.000000
13  Tomcat03     log.row_count              2  2021-03-07 03:10:00+08:00                15.00      7.8000        1.923077
14  Tomcat04  log.gc_pause_max              1  2021-03-07 03:03:00+08:00                 0.07      0.0695        1.007194
15  Tomcat03  log.gc_pause_max              1  2021-03-07 03:08:00+08:00                 0.07      0.0690        1.014493
16   Mysql01   log.error_count              1  2021-03-07 03:24:00+08:00                 0.00      0.0000             NaN
17   Mysql01     log.row_count              1  2021-03-07 03:24:00+08:00                 1.00      1.0000        1.000000, Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel: metric, global_p95 (thresholds precomputed)
tz = pytz.timezone('Asia/Shanghai')

# Incident window in UTC+8 -> epoch seconds (UTC) (inclusive)
start_local = pd.Timestamp('2021-03-07 03:00:00', tz=tz)
end_local   = pd.Timestamp('2021-03-07 03:30:00', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts   = int(end_local.tz_convert('UTC').timestamp())

# 1) Filter metric.csv to the incident window
window_metric = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 2) Merge with existing global_p95 thresholds and mark anomalies (value >= global_p95)
# (Do NOT recalculate thresholds here)
merged = window_metric.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
# Drop rows without a threshold (no global_p95)
merged = merged.dropna(subset=['global_p95']).copy()
merged['is_anomaly'] = merged['value'] >= merged['global_p95']

# Keep only anomalous rows for run detection
anoms = merged[merged['is_anomaly']].sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

# Prepare empty results if no anomalies
if anoms.empty:
    top20_fault_runs = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'duration_minutes','points_in_fault','max_value_in_fault','global_p95','severity_ratio'
    ])
    earliest_high_sev = top20_fault_runs.head(1)
else:
    # 3) Collapse consecutive anomalous minutes into fault runs per (cmdb_id, kpi_name)
    anoms['prev_ts'] = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].shift(1)
    anoms['ts_diff'] = anoms['timestamp'] - anoms['prev_ts']
    anoms['new_run'] = (anoms['ts_diff'] != 60) | (anoms['ts_diff'].isna())
    anoms['run_id'] = anoms.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()

    # 4) Aggregate each run
    agg = anoms.groupby(['cmdb_id','kpi_name','run_id'], as_index=False).agg(
        fault_start_ts=('timestamp','min'),
        fault_end_ts=('timestamp','max'),
        points_in_fault=('timestamp','size'),
        max_value_in_fault=('value','max'),
        global_p95=('global_p95','first')
    )
    # duration in minutes (inclusive)
    agg['duration_minutes'] = ((agg['fault_end_ts'] - agg['fault_start_ts']) // 60) + 1
    # severity ratio, safe divide (global_p95 == 0 -> NaN)
    agg['severity_ratio'] = np.where(agg['global_p95'] == 0, np.nan, agg['max_value_in_fault'] / agg['global_p95'])

    # convert timestamps to tz-aware datetimes (Asia/Shanghai)
    agg['fault_start_timestamp'] = pd.to_datetime(agg['fault_start_ts'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    agg['fault_end_timestamp']   = pd.to_datetime(agg['fault_end_ts'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

    # Select and order columns as requested
    fault_runs = agg[['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                      'duration_minutes','points_in_fault','max_value_in_fault','global_p95','severity_ratio']].copy()

    # Round numeric columns for compactness
    fault_runs[['max_value_in_fault','global_p95','severity_ratio']] = fault_runs[['max_value_in_fault','global_p95','severity_ratio']].round(6)

    # 5A) Top 20 fault runs across all components sorted by severity_ratio desc (NaNs last), then start asc
    # Use na_position='last' so NaN severity are treated as lowest severity
    top20_fault_runs = fault_runs.sort_values(
        by=['severity_ratio','fault_start_timestamp'],
        ascending=[False, True],
        na_position='last'
    ).head(20).reset_index(drop=True)

    # 5B) Earliest fault run among runs whose severity_ratio >= 1.5; if none, earliest run overall
    high_sev = fault_runs[fault_runs['severity_ratio'] >= 1.5].copy()
    if not high_sev.empty:
        earliest_high_sev = high_sev.sort_values('fault_start_timestamp', ascending=True).head(1).reset_index(drop=True)
    else:
        earliest_high_sev = fault_runs.sort_values('fault_start_timestamp', ascending=True).head(1).reset_index(drop=True)

# Return the two compact outputs (DataFrames)
top20_fault_runs, earliest_high_sev

```
Out[6]:
```
```python
# Compose a concise plain-English summary string based on existing kernel variables:
# top20_fault_runs and earliest_high_sev (computed previously).
# Reuse these variables and present a compact summary.

if 'top20_fault_runs' not in globals() or top20_fault_runs.empty:
    summary = "No fault runs detected in the incident window."
else:
    df = top20_fault_runs.copy()
    # Top 3 fault runs by severity (already sorted)
    top3 = df.head(3)
    top3_lines = []
    for i, r in top3.iterrows():
        sev = r['severity_ratio']
        sev_str = "NaN" if pd.isna(sev) else f"{sev:.3f}"
        top3_lines.append(
            f"{i+1}) {r['cmdb_id']} | {r['kpi_name']} | {r['fault_start_timestamp']} -> {r['fault_end_timestamp']} | "
            f"duration={int(r['duration_minutes'])}min | points={int(r['points_in_fault'])} | max={r['max_value_in_fault']:.6g} | p95={r['global_p95']:.6g} | sev={sev_str}"
        )
    # Earliest severe run chosen (earliest run with severity>=1.5 or earliest overall)
    if 'earliest_high_sev' in globals() and not earliest_high_sev.empty:
        e = earliest_high_sev.iloc[0]
        e_sev = e['severity_ratio']
        e_sev_str = "NaN" if pd.isna(e_sev) else f"{e_sev:.3f}"
        earliest_line = (
            f"{e['cmdb_id']} | {e['kpi_name']} | {e['fault_start_timestamp']} -> {e['fault_end_timestamp']} | "
            f"duration={int(e['duration_minutes'])}min | points={int(e['points_in_fault'])} | max={e['max_value_in_fault']:.6g} | p95={e['global_p95']:.6g} | sev={e_sev_str}"
        )
    else:
        earliest_line = "No fault runs available."

    # Compose final summary
    summary = (
        "Fault-run analysis (incident window 2021-03-07 03:00–03:30 +08:00):\n\n"
        "1) Top fault runs by severity (examples):\n  " + "\n  ".join(top3_lines) + "\n\n"
        "2) Key observations:\n"
        "- MySQL (Mysql01/Mysql02) dominates the highest-severity runs. Examples include 'Sort Scan' and 'Rows Read' on Mysql01 with severity ratios up to ~9.7x and ~9.5x, and several InnoDB I/O/write metrics with ~8.9x.\n"
        "- Many high-severity runs are short (1–2 minutes) and have few points, indicating brief but large spikes rather than prolonged sustained faults.\n"
        "- Other notable high-severity items include disk I/O metrics (sda/sdb/sdc) with severity ~3–4x, and a Tomcat CPU WIO spike (~3.95x).\n\n"
        "3) Earliest severe run (severity >=1.5 if any; otherwise earliest overall):\n  " + earliest_line + "\n\n"
        "Actionable guidance:\n"
        "- Prioritize investigating Mysql01/Mysql02 I/O and query activity around 03:00–03:12 (sorts, rows-read, InnoDB writes, buffer pool flushes).\n"
        "- Check underlying disk performance on devices sda/sdb/sdc and any bursty workload causing short I/O spikes.\n"
        "- Because many runs are brief, correlate with request traces and logs for those minutes to identify root cause events (heavy queries, backups, or batch jobs).\n"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name     fault_start_timestamp       fault_end_timestamp  duration_minutes  points_in_fault  max_value_in_fault  global_p95  severity_ratio
0    Mysql01                         Mysql-MySQL_3306_Sort Scan 2021-03-07 03:11:00+08:00 2021-03-07 03:12:00+08:00                 2                2            0.163900     0.01690        9.698225
1    Mysql01                         Mysql-MySQL_3306_Rows Read 2021-03-07 03:11:00+08:00 2021-03-07 03:12:00+08:00                 2                2           87.754100     9.27120        9.465236
2    Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages flushed 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.783300     0.20000        8.916500
3    Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.783300     0.20000        8.916500
4    Mysql01              Mysql-MySQL_3306_Innodb pages written 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1            1.783300     0.20000        8.916500
5    Mysql01               Mysql-MySQL_3306_Innodb data written 2021-03-07 03:01:00+08:00 2021-03-07 03:01:00+08:00                 1                1        58470.400000  6826.66670        8.565000
6    Mysql02                 Mysql-MySQL_3306_Handler Read Prev 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1           19.433300     3.76670        5.159238
7   apache01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer... 2021-03-07 03:00:00+08:00 2021-03-07 03:00:00+08:00                 1                1            0.400000     0.10000        4.000000
8       IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-07 03:14:00+08:00 2021-03-07 03:14:00+08:00                 1                1            0.050000     0.01250        4.000000
9       IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-07 03:14:00+08:00 2021-03-07 03:14:00+08:00                 1                1            0.400000     0.10000        4.000000
10   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKBps 2021-03-07 03:28:00+08:00 2021-03-07 03:28:00+08:00                 1                1            5.000000     1.25000        4.000000
11  Tomcat02                             OSLinux-CPU_CPU_CPUWio 2021-03-07 03:17:00+08:00 2021-03-07 03:17:00+08:00                 1                1            0.195300     0.04941        3.952641
12   Mysql01         Mysql-MySQL_3306_Innodb log write requests 2021-03-07 03:00:00+08:00 2021-03-07 03:00:00+08:00                 1                1            2.733300     0.70000        3.904714
13   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-07 03:02:00+08:00 2021-03-07 03:02:00+08:00                 1                1            5.000000     1.50000        3.333333
14   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-07 03:15:00+08:00 2021-03-07 03:15:00+08:00                 1                1            0.800000     0.24000        3.333333
15   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.016667     0.00500        3.333333
16   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.133333     0.04000        3.333333
17   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.016700     0.00501        3.333333
18   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-07 03:23:00+08:00 2021-03-07 03:23:00+08:00                 1                1            0.133300     0.03999        3.333333
19   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-07 03:27:00+08:00 2021-03-07 03:27:00+08:00                 1                1            0.533300     0.15999        3.333333,    cmdb_id                                kpi_name     fault_start_timestamp       fault_end_timestamp  duration_minutes  points_in_fault  max_value_in_fault  global_p95  severity_ratio
0  Mysql01  Mysql-MySQL_3306_Innodb os log written 2021-03-07 03:00:00+08:00 2021-03-07 03:00:00+08:00                 1                1              1510.4    657.0667        2.298701)```
```